In [11]:
from google.colab import drive
from pathlib import Path

MOUNT_POINT = "/content/gdrive"

drive.mount(MOUNT_POINT)

PROJECT_DIR = Path(MOUNT_POINT) / "MyDrive" / "Underwater-Image-Data-set-main"

print("Project exists:", PROJECT_DIR.exists())
print("Project path:", PROJECT_DIR)

Mounted at /content/gdrive
Project exists: True
Project path: /content/gdrive/MyDrive/Underwater-Image-Data-set-main


In [12]:
from pathlib import Path
import pandas as pd
import json
import shutil

V1_DIR = PROJECT_DIR / "Dataset_V1"

FINAL_DIR = V1_DIR / "Final_Method"
REVIEW_DIR = FINAL_DIR / "Final_Review"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_FILE = FINAL_DIR / "FINAL_PIPELINE_CONFIG.json"
RATIONALE_FILE = FINAL_DIR / "final_method_selection_rationale.json"
EVAL_DIR = FINAL_DIR / "Final_Evaluation"

with open(CONFIG_FILE, "r") as f:
    final_config = json.load(f)

with open(RATIONALE_FILE, "r") as f:
    rationale = json.load(f)

final_method = final_config["final_method"]
final_parameters = final_config["parameters"]

split_config = final_config["split"]
preprocessing_config = final_config["preprocessing"]
evaluation_config = final_config["evaluation"]

print("=" * 70)
print("FINAL REVIEW PACKAGE")
print("=" * 70)

print("\nFinal method:", final_method)
print("Final parameters:", final_parameters)

summary_file = (
    EVAL_DIR /
    "final_test_results_summary.csv"
)

if summary_file.exists():

    final_results = pd.read_csv(
        summary_file
    )

    final_results.to_csv(
        REVIEW_DIR /
        "final_results_for_review.csv",
        index=False
    )

    print("\nFINAL RESULTS")
    print(final_results)

else:

    final_results = pd.DataFrame()

    print("\nFinal evaluation results not found.")

if EVAL_DIR.exists():

    tradeoff_file = (
        EVAL_DIR /
        "quality_complexity_tradeoff.json"
    )

    if tradeoff_file.exists():

        with open(
            tradeoff_file,
            "r"
        ) as f:

            tradeoff = json.load(f)

    else:

        tradeoff = {}

else:

    tradeoff = {}

workflow = {
    "step_1_dataset": "Dataset V1 frozen split",
    "step_2_split": "70% train, 15% validation, 15% test",
    "step_3_unit_of_split": "sample_id = source_archive + image_id",
    "step_4_triplet_handling": "Input, Target and Generated remain together",
    "step_5_preprocessing": preprocessing_config,
    "step_6_method_selection": "Selected using validation evidence",
    "step_7_parameter_tuning": "Performed only on validation data",
    "step_8_model_or_pipeline": {
        "method": final_method,
        "parameters": final_parameters
    },
    "step_9_final_evaluation": "Frozen held-out test set",
    "step_10_metrics": evaluation_config["primary_metrics"],
    "step_11_additional_metrics": evaluation_config["additional_metrics"],
    "step_12_outputs": "Final enhanced outputs and evaluation tables"
}

with open(
    REVIEW_DIR /
    "reproducible_input_to_output_workflow.json",
    "w"
) as f:

    json.dump(
        workflow,
        f,
        indent=4
    )

remaining_questions = pd.DataFrame({
    "Question": [
        "Which failure cases remain difficult for the final method?",
        "Are there cases where PSNR/SSIM disagree with visual quality?",
        "Does the final method introduce color or contrast artifacts?",
        "Are edge-preservation failures concentrated in dark or blurry images?",
        "Is processing time acceptable for the intended use case?",
        "Is an additional learning-based model required?",
        "Are UIQM/UCIQE required in the final report?"
    ],
    "Status": [
        "To review",
        "To review",
        "To review",
        "To review",
        "To review",
        "To discuss",
        "To confirm"
    ],
    "Notes": [
        "",
        "",
        "",
        "",
        "",
        "",
        ""
    ]
})

remaining_questions.to_csv(
    REVIEW_DIR /
    "remaining_questions.csv",
    index=False
)

error_requirements = pd.DataFrame({
    "Requirement": [
        "Representative successful cases",
        "Representative failure cases",
        "Dark-image failure analysis",
        "Blur/sharpness failure analysis",
        "Color/contrast artifact analysis",
        "Edge-preservation analysis",
        "Metric disagreement analysis",
        "Processing-time analysis",
        "Final limitations documented"
    ],
    "Completed": [
        "Review",
        "Review",
        "Review",
        "Review",
        "Review",
        "Review",
        "Review",
        "Review",
        "Review"
    ],
    "Notes": [
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        ""
    ]
})

error_requirements.to_csv(
    REVIEW_DIR /
    "final_error_analysis_requirements.csv",
    index=False
)

report_requirements = pd.DataFrame({
    "Reporting Item": [
        "Final method and parameters",
        "Frozen preprocessing",
        "Frozen train/validation/test split",
        "Validation-based selection rationale",
        "Final held-out test results",
        "Baseline comparison",
        "Edge-preservation analysis",
        "Runtime/processing time",
        "Model/pipeline size",
        "Success and failure examples",
        "Remaining limitations",
        "Reproducibility information"
    ],
    "Status": [
        "Ready",
        "Ready",
        "Ready",
        "Ready",
        "Ready",
        "Ready",
        "Ready",
        "Ready",
        "Ready",
        "To review",
        "To review",
        "Ready"
    ]
})

report_requirements.to_csv(
    REVIEW_DIR /
    "final_reporting_requirements.csv",
    index=False
)

mentor_corrections = pd.DataFrame({
    "Date": [""],
    "Mentor Correction": [""],
    "Action Required": [""],
    "Status": ["Pending"],
    "Completed": [False]
})

mentor_corrections.to_csv(
    REVIEW_DIR /
    "mentor_corrections.csv",
    index=False
)

review_summary = {
    "final_method": final_method,
    "final_parameters": final_parameters,
    "validation_based_selection": True,
    "preprocessing_frozen": True,
    "split_frozen": True,
    "evaluation_methodology_frozen": True,
    "final_test_evaluation_completed":
        len(final_results) > 0,
    "reproducible_workflow_documented": True,
    "remaining_questions_documented": True,
    "error_analysis_requirements_documented": True,
    "reporting_requirements_documented": True,
    "mentor_corrections_record_created": True,
    "mentor_corrections_recorded":
        False
}

with open(
    REVIEW_DIR /
    "FINAL_REVIEW_SUMMARY.json",
    "w"
) as f:

    json.dump(
        review_summary,
        f,
        indent=4
    )

print("\n" + "=" * 70)
print("FINAL REVIEW PACKAGE CREATED")
print("=" * 70)

print("\nFinal method:")
print(final_method)

print("\nWorkflow documented:")
print("YES")

print("\nRemaining questions documented:")
print("YES")

print("\nError-analysis requirements documented:")
print("YES")

print("\nReporting requirements documented:")
print("YES")

print("\nMentor corrections:")
print("Template created — actual corrections must be entered after mentor review.")

print("\nFiles created:")

for file in sorted(
    REVIEW_DIR.iterdir()
):

    print(
        " -",
        file.name
    )

print("\nReview folder:")
print(REVIEW_DIR)

FINAL REVIEW PACKAGE

Final method: Gamma
Final parameters: {'gamma': 0.8}

FINAL RESULTS
               method       PSNR      SSIM  Edge Preservation  Edge Precision  \
0  Final Tuned Method  13.844543  0.654768           0.007723        0.025546   
1      Input Baseline  13.365501  0.654419           0.008117        0.025219   
2               U-Net  14.162797  0.630716           0.032845        0.014136   

   Edge Recall   Edge F1  Processing Time  
0     0.007723  0.010610         0.004036  
1     0.008117  0.010812         0.000045  
2     0.032845  0.017006         0.330070  

FINAL REVIEW PACKAGE CREATED

Final method:
Gamma

Workflow documented:
YES

Remaining questions documented:
YES

Error-analysis requirements documented:
YES

Reporting requirements documented:
YES

Mentor corrections:
Template created — actual corrections must be entered after mentor review.

Files created:
 - FINAL_REVIEW_SUMMARY.json
 - final_error_analysis_requirements.csv
 - final_reporting_requireme